In [2]:
import os
import faiss
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Read from ingested KB (hybrid: generated + manual)
kb_dir = Path("../kb/ingested")

# Create vector_store directory if needed
os.makedirs("../vector_store", exist_ok=True)

kb_docs = []
kb_sources = []

if kb_dir.exists():
    print(f"Loading KB from: {kb_dir}")
    
    for file_path in sorted(kb_dir.glob("*.md")):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
            
            kb_docs.append(content)
            kb_sources.append(file_path.name)
            print(f"  ✓ Loaded: {file_path.name}")
        
        except Exception as e:
            print(f"  ✗ Error loading {file_path.name}: {e}")
    
    print(f"\nTotal KB articles loaded: {len(kb_docs)}\n")
else:
    print(f"❌ KB directory not found: {kb_dir}")
    print("Please run kb_creation.ipynb first to generate KB articles.")

if kb_docs:
    print("Generating embeddings...")
    kb_embeddings = embedder.encode(
        kb_docs,
        show_progress_bar=True,
        convert_to_numpy=True
    ).astype("float32")

    print(f"Building FAISS index...")
    kb_index = faiss.IndexFlatL2(kb_embeddings.shape[1])
    kb_index.add(kb_embeddings)

    faiss.write_index(kb_index, "../vector_store/kb_faiss.index")
    print(f"✓ Saved FAISS index: ../vector_store/kb_faiss.index")

    kb_metadata = [
        {
            "source": source,
            "content": content
        }
        for source, content in zip(kb_sources, kb_docs)
    ]

    with open("../vector_store/kb_metadata.pkl", "wb") as f:
        pickle.dump(kb_metadata, f)
    print(f"✓ Saved KB metadata: ../vector_store/kb_metadata.pkl")
    
    print(f"\n{'='*50}")
    print(f"KB Indexing Complete!")
    print(f"Total articles indexed: {len(kb_docs)}")
    print(f"{'='*50}")
else:
    print("No KB articles found. Cannot create FAISS index.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading KB from: ..\kb\ingested
  ✓ Loaded: access_issues_and_troubleshooting_guide_ingested.md
  ✓ Loaded: direct_reports_and_time_card_issues_ingested.md
  ✓ Loaded: generated_access_issues_and_troubleshooting_guide.md
  ✓ Loaded: generated_direct_reports_and_time_card_issues.md
  ✓ Loaded: generated_hardware_and_access_connection_issues.md
  ✓ Loaded: generated_meeting_request_and_cable_issues.md
  ✓ Loaded: generated_network_access_issues.md
  ✓ Loaded: generated_new_purchase_po_support.md
  ✓ Loaded: generated_new_starter_onboarding_process.md
  ✓ Loaded: generated_workspace_configuration_issues.md
  ✓ Loaded: hardware_and_access_connection_issues_ingested.md
  ✓ Loaded: manual_hardware_request_process.md
  ✓ Loaded: manual_microsoft_365_login_issue.md
  ✓ Loaded: manual_password_reset_not_working.md
  ✓ Loaded: manual_vpn_connection_issues.md
  ✓ Loaded: meeting_request_and_cable_issues_ingested.md
  ✓ Loaded: network_access_issues_ingested.md
  ✓ Loaded: new_purchase_po_support_

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Building FAISS index...
✓ Saved FAISS index: ../vector_store/kb_faiss.index
✓ Saved KB metadata: ../vector_store/kb_metadata.pkl

KB Indexing Complete!
Total articles indexed: 20


In [3]:
def retrieve_kb(query, top_k=3):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = kb_index.search(query_embedding, top_k)

    results = []

    for idx, dist in zip(indices[0], distances[0]):
        item = kb_metadata[idx].copy()
        item["distance"] = float(dist)
        results.append(item)

    return results

In [4]:
import requests

def call_ollama(prompt, model="llama3.2:3b"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        },
        timeout=120
    )

    response.raise_for_status()

    return response.json()["response"]

In [5]:
def generate_resolution(ticket, category, priority, similar_tickets, kb_results):
    similar_context = "\n\n".join([
        f"- Category: {item['category']}, Priority: {item['priority']}, Ticket: {item['ticket_text']}"
        for item in similar_tickets
    ])

    kb_context = "\n\n".join([
        f"Source: {item['source']}\n{item['content']}"
        for item in kb_results
    ])

    prompt = f"""
You are an IT service desk assistant for an enterprise managed service provider.

Use the ticket, predicted category, predicted priority, similar past tickets, and knowledge base content to generate a concise troubleshooting recommendation.

Ticket:
{ticket}

Predicted Category:
{category}

Predicted Priority:
{priority}

Similar Past Tickets:
{similar_context}

Knowledge Base:
{kb_context}

Output format:
1. Issue Summary
2. Recommended Troubleshooting Steps
3. Suggested Escalation Team
4. Risk Note
5. Sources Used
"""

    return call_ollama(prompt)